# Class 6 Challenge — Can this data safely support a business decision?

Leadership claim: **“Late Delivery Rate is 56%.”**

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL / UNKNOWN → publish decision.**
These checks became the executable gate in `pipeline/validate.py` (V01–V24).

In [1]:
import json, sqlite3
from pathlib import Path
import pandas as pd
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = BASE / "source_systems"
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 120)
con = sqlite3.connect(SRC / "database" / "flasheats.db")
orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)
tickets = pd.read_csv(SRC / "exports" / "support_tickets.csv")
restaurant_status = pd.read_csv(SRC / "exports" / "restaurant_status.csv")
metric_notes = json.loads((SRC / "client_docs" / "client_metric_definitions.json").read_text())
de = json.loads((SRC / "exports" / "driver_events.json").read_text())
events = pd.DataFrame([{"driver_id": d["driver_id"], **e} for d in de for e in d["events"]])
ts = lambda s: pd.to_datetime(s, format="mixed", errors="coerce")
print(orders.shape, tickets.shape, restaurant_status.shape); print(json.dumps(metric_notes, indent=2))

(1603, 13) (202, 5) (502, 4)
{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?

| Business assumption | Data expectation | How we test it | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` unique | duplicate count + do duplicates disagree? | WARN if rare & agreed dedupe rule; FAIL if systemic |
| Delivered orders have completion time | delivered ⇒ `actual_delivery_at` not null | null count; can another source fill it? | WARN (denominator shrinks, biased) |
| Promised ETA is valid | `promised_eta ≥ created_at` | chronology rule | WARN — exclude impossible promises |
| Event chronology is valid | created ≤ pickup ≤ delivered | chronology rules, cross-checked with Driver App | WARN — use the system of record |
| “Late” has an agreed definition | one owner, one formula | read client definitions | UNKNOWN until an owner signs |

In [2]:
o = orders.copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]: o[c + "_t"] = ts(o[c])
dups = o[o.order_id.duplicated(keep=False)].sort_values("order_id")
print("rows", len(o), "| unique", o.order_id.nunique(), "| extra duplicate rows", len(o) - o.order_id.nunique())
display(dups[["order_id", "created_at", "final_status", "traffic_bucket", "distance_km_estimate"]])
u = o.drop_duplicates("order_id").copy()
st = u.final_status.str.strip().str.lower()
checks = {
    "delivered but no actual_delivery_at": int(((st == "delivered") & u.actual_delivery_at_t.isna()).sum()),
    "promised_eta < created_at": int((u.promised_eta_t < u.created_at_t).sum()),
    "pickup_at < created_at": int((u.pickup_at_t < u.created_at_t).sum()),
    "actual_delivery_at < pickup_at": int((u.actual_delivery_at_t < u.pickup_at_t).sum()),
    "cancelled but has pickup_at": int(((st == "cancelled") & u.pickup_at.notna()).sum()),
}
checks

rows 1603 | unique 1600 | extra duplicate rows 3


,order_id,created_at,final_status,traffic_bucket,distance_km_estimate
119,O00120,2026-08-13T18:15:00,delivered,severe,18.00
1600,O00120,2026-08-13T18:15:00,delivered,medium,18.00
722,O00723,2026-08-05T21:04:00,delivered,medium,18.00
1601,O00723,2026-08-05T21:04:00,delivered,high,18.00
1301,O01302,2026-08-26T21:07:00,delivered,medium,14.78
1602,O01302,2026-08-26T21:07:00,delivered,high,14.78


{'delivered but no actual_delivery_at': 37,
 'promised_eta < created_at': 4,
 'pickup_at < created_at': 0,
 'actual_delivery_at < pickup_at': 5,
 'cancelled but has pickup_at': 68}

In [3]:
display(u[u.promised_eta_t < u.created_at_t][["order_id", "created_at", "promised_eta"]])
display(u[u.actual_delivery_at_t < u.pickup_at_t][["order_id", "created_at", "pickup_at", "actual_delivery_at"]])
life = events[events.type != "gps_ping"].assign(t=ts(events.timestamp)).pivot_table(index="order_id", columns="type", values="t", aggfunc="first")
bad = u[u.actual_delivery_at_t < u.pickup_at_t].join(life, on="order_id")
print("Driver App pickup for those 5 orders is before delivery:", (bad.picked_up < bad.actual_delivery_at_t).all())
print("cancelled orders with a Driver App pickup:", u[st == "cancelled"].order_id.isin(life.dropna(subset=["picked_up"]).index).sum())

,order_id,created_at,promised_eta
100,O00101,2026-08-24T20:59:00,2026-08-24T20:49:00
101,O00102,2026-08-11T16:41:00,2026-08-11T16:31:00
102,O00103,2026-08-22T20:11:00,2026-08-22T20:01:00
103,O00104,2026-08-09T22:19:00,2026-08-09T22:09:00


,order_id,created_at,pickup_at,actual_delivery_at
50,O00051,2026-08-01T15:09:00,2026-08-01T16:24:51.495665,2026-08-01T16:19:51.495665
51,O00052,2026-08-24T17:07:00,2026-08-24T18:16:16.946011,2026-08-24T18:11:16.946011
52,O00053,2026-08-13T17:06:00,2026-08-13T18:25:53.797613,2026-08-13T18:20:53.797613
53,O00054,2026-08-28T19:56:00,2026-08-28T21:25:10.582862,2026-08-28T21:20:10.582862
54,O00055,2026-08-16T12:53:00,2026-08-16T14:08:05.645770,2026-08-16T14:03:05.645770


Driver App pickup for those 5 orders is before delivery: True
cancelled orders with a Driver App pickup: 0


**Findings.** The duplicates are **not identical** — they disagree on `traffic_bucket`, so “keep first” is a business rule that needs Order Platform sign-off, not a free cleanup. 4 promises are made 10 minutes *before* the order exists; 5 DB pickups come after delivery (the Driver App's pickup is valid → use it); all 68 cancelled orders carry a DB `pickup_at` although the Driver App never picked them up (the field is a placeholder for cancelled orders). **The 56% cannot be defended as a single number until the definition is owned.**

# Challenge 2 — Stakeholders disagree on “late”

In [4]:
de_ = u[(st == "delivered")].copy()
de_["delay"] = (de_.actual_delivery_at_t - de_.promised_eta_t).dt.total_seconds() / 60
hist = de_.dropna(subset=["delay"])
valid_promise = hist[hist.promised_eta_t >= hist.created_at_t]
tbl = pd.DataFrame([
    ["VP Operations: any delay > 0", len(hist), round(100*(hist.delay > 0).mean(), 1), "every broken promise counts"],
    ["Support Lead: delay > 10 min", len(hist), round(100*(hist.delay > 10).mean(), 1), "only delays customers call about"],
    ["Finance: exclude cancelled/refunded", len(hist), round(100*(hist.delay > 0).mean(), 1), "cancelled already outside the population"],
    ["Data Team: delivered + non-null actual (historical)", len(hist), round(100*(hist.delay > 0).mean(), 1), "silently drops the 37 missing completions"],
    ["…and removing impossible promises", len(valid_promise), round(100*(valid_promise.delay > 0).mean(), 1), "4 orders promised before creation"],
], columns=["Definition", "Orders", "Late rate %", "Business meaning"])
tbl

,Definition,Orders,Late rate %,Business meaning
0,VP Operations: any delay > 0,1495,56.4,every broken promise counts
1,Support Lead: delay > 10 min,1495,23.3,only delays customers call about
2,Finance: exclude cancelled/refunded,1495,56.4,cancelled already outside the population
3,Data Team: delivered + non-null actual (historical),1495,56.4,silently drops the 37 missing completions
4,…and removing impossible promises,1491,56.3,4 orders promised before creation


**Which one should leadership publish, and who owns it?** Publish the **VP Operations** definition (any delay after the promise — it is what the customer experiences) as the headline, *always shown next to* the Support Lead's > 10-minute rate (23%), because 56% vs 23% is a different business problem. The client note says **no canonical KPI owner exists** → the VP Operations must sign the definition; until then the metric is published with an explicit ⚠️ caveat (pipeline check **V23 = UNKNOWN**).

# Challenge 3 — Validate categories without cleaning by instinct

In [5]:
for name, s in [("final_status", orders.final_status), ("traffic_bucket", orders.traffic_bucket), ("restaurant status", restaurant_status.status), ("ticket category", tickets.category)]:
    print(f"\n{name}:", s.value_counts(dropna=False).to_dict())


final_status: {'delivered': 1530, 'cancelled': 68, 'Delivered': 5}

traffic_bucket: {'medium': 635, 'high': 448, 'low': 381, 'severe': 136, 'HIGH': 3}

restaurant status: {'preparing': 171, 'handed_off': 168, 'ready': 157, 'ready ': 2, 'READY': 1, 'Ready': 1, 'handoff': 1, 'unknown': 1}

ticket category: {'late_delivery': 38, 'eta_changed': 37, 'restaurant_delay': 37, 'ready_but_waiting': 33, 'status_mismatch': 29, 'driver_not_moving': 25, 'Late Delivery': 1, 'late_delivery ': 1, 'ETA issue': 1}


| Field | Observed variants | Safe to normalise (representation) | Needs owner confirmation (semantics) |
|---|---|---|---|
| `final_status` | delivered / Delivered / cancelled | `Delivered → delivered` (case) | — |
| `traffic_bucket` | low, medium, high, **HIGH**, severe | `HIGH → high` (case) | — |
| restaurant `status` | preparing, ready, Ready, READY, `ready␠`, handed_off, **handoff**, **unknown** | case + trailing space → `ready` | `handoff` ≟ `handed_off` (another client app?); `unknown` stays unknown |
| ticket `category` | late_delivery, `late_delivery␠`, **Late Delivery**, eta_changed, **ETA issue** … | case/space/separator → `late_delivery` | `ETA issue` ≟ `eta_changed` — “issue” is broader than “changed” |

The agreed mapping lives in `config/category_map.csv` with an `owner_confirmed` column; unmapped values surface as `UNMAPPED:<raw>` rather than being guessed.

# Challenge 4 — Cross-source integrity

In [6]:
oid = set(u.order_id)
rels = [
    ("orders.restaurant_id → restaurants", orders.restaurant_id.dropna().isin(restaurants.restaurant_id).mean(), orders.restaurant_id.isna().sum(), "per-restaurant cuts"),
    ("orders.driver_id → drivers", orders.driver_id.dropna().isin(drivers.driver_id).mean(), orders.driver_id.isna().sum(), "per-driver cuts"),
    ("tickets.order_id → orders", tickets.order_id.dropna().isin(oid).mean(), tickets.order_id.isna().sum(), "ticket ↔ outcome join"),
    ("restaurant_status.order_id → orders", restaurant_status.order_id.isin(oid).mean(), 0, "readiness evidence"),
    ("driver_events.order_id → orders", events.order_id.isin(oid).mean(), 0, "pickup/delivery evidence"),
]
integ = pd.DataFrame([[r, round(100*c, 2), int(n), "PASS" if c == 1 and n == 0 else "WARN", risk] for r, c, n, risk in rels],
                     columns=["Relationship", "Coverage % (non-null keys)", "Null keys", "Status", "Risk if broken"])
display(integ)
bb = restaurants[~(restaurants.lat.between(12.6, 13.4) & restaurants.lon.between(77.2, 78.0))]
print("restaurants with impossible coordinates:"); display(bb)
print("restaurant_status covers", restaurant_status.order_id.nunique(), "of", len(oid), "orders")

,Relationship,Coverage % (non-null keys),Null keys,Status,Risk if broken
0,orders.restaurant_id → restaurants,100.0,3,WARN,per-restaurant cuts
1,orders.driver_id → drivers,100.0,3,WARN,per-driver cuts
2,tickets.order_id → orders,100.0,3,WARN,ticket ↔ outcome join
3,restaurant_status.order_id → orders,100.0,0,PASS,readiness evidence
4,driver_events.order_id → orders,100.0,0,PASS,pickup/delivery evidence


restaurants with impossible coordinates:


,restaurant_id,restaurant_name,cuisine,lat,lon,manual_status_updates
6,R007,Restaurant 007,North Indian,95.120000,77.694095,0
18,R019,Restaurant 019,Cafe,12.847954,190.330000,1


restaurant_status covers 500 of 1600 orders


**Is 1% unmapped acceptable?** It depends on the decision: 3 orders with a null `restaurant_id` do not move the city-wide KPI (0.2%), but they would silently vanish from a per-restaurant league table used to penalise partners. All keys that exist map 100% — the risk is **nulls and coverage**, not wrong keys (restaurant status exists for only 500 of 1,600 orders).

# Challenge 5 — Freshness is an SLA question

In [7]:
rs = restaurant_status.drop_duplicates().copy()
rs["status_n"] = rs.status.str.strip().str.lower().replace({"handoff": "handed_off"})
rs["upd"] = ts(rs.last_updated_at)
rs = rs.join(life, on="order_id").merge(u[["order_id", "created_at_t"]], on="order_id")
rs["upd_vs_pickup_min"] = (rs.upd - rs.picked_up).dt.total_seconds() / 60
display(rs.groupby("status_n").upd_vs_pickup_min.describe()[["count", "25%", "50%", "75%"]].round(1))
print("'preparing'/'ready' updated AFTER the driver picked up:", int((rs.status_n.isin(["preparing", "ready"]) & (rs.upd_vs_pickup_min > 0)).sum()))
print("minute-rounded timestamps:", (rs.upd.dt.second == 0).mean())

,count,25%,50%,75%
status_n,,,,
handed_off,164.0,-14.7,-6.0,3.2
preparing,164.0,-16.5,-7.6,1.7
ready,156.0,-17.2,-5.7,5.2
unknown,1.0,-3.2,-3.2,-3.2


'preparing'/'ready' updated AFTER the driver picked up: 104
minute-rounded timestamps: 1.0


| Use case | Freshness / fidelity needed | Is `restaurant_status.csv` fit? |
|---|---|---|
| Weekly analytics | daily batch is fine, but needs coverage | ⚠️ WARN — covers ~31% of orders, one snapshot each |
| Live customer ETA | seconds–minutes, every state change | ❌ FAIL — last-updated snapshot, minute-rounded, manual |
| Restaurant accountability | complete event history per order | ❓ UNKNOWN — 100+ “preparing/ready” updates arrive after pickup; cannot prove when food was ready |

Same record, three different verdicts — freshness is a property of the *decision*, not of the file.

# Challenge 6 — The validation gate

In [8]:
validation_report = {
    "business_grain": ("WARN", "3 conflicting duplicates (0.19%) — keep-first rule pending Order Platform sign-off"),
    "timestamp_chronology": ("WARN", "4 impossible promises excluded; 5 DB pickups replaced by Driver App; cancelled pickup_at ignored"),
    "kpi_definition": ("UNKNOWN", "4 stakeholder definitions, no owner (56.4% vs 23.3%)"),
    "category_semantics": ("WARN", "case/space normalised; handoff & 'ETA issue' mapped provisionally, need owners"),
    "cross_source_mapping": ("WARN", "100% of non-null keys map; 3 null restaurant_id, 3 null driver_id, 3 unlinked tickets"),
    "freshness": ("PASS for weekly KPI / FAIL for live ETA", "restaurant status unfit for live or accountability use"),
    "publish_56_percent": ("PUBLISH WITH CAVEAT", "publish '56.5% (VP Ops def., 1,528 orders; 23.2% > 10 min)' — not a bare '56%'"),
}
pd.DataFrame(validation_report, index=["Status", "Evidence / action"]).T

,Status,Evidence / action
business_grain,WARN,3 conflicting duplicates (0.19%) — keep-first rule pending Order Platform sign-off
timestamp_chronology,WARN,4 impossible promises excluded; 5 DB pickups replaced by Driver App; cancelled pickup_at ignored
kpi_definition,UNKNOWN,"4 stakeholder definitions, no owner (56.4% vs 23.3%)"
category_semantics,WARN,"case/space normalised; handoff & 'ETA issue' mapped provisionally, need owners"
cross_source_mapping,WARN,"100% of non-null keys map; 3 null restaurant_id, 3 null driver_id, 3 unlinked tickets"
freshness,PASS for weekly KPI / FAIL for live ETA,restaurant status unfit for live or accountability use
publish_56_percent,PUBLISH WITH CAVEAT,"publish '56.5% (VP Ops def., 1,528 orders; 23.2% > 10 min)' — not a bare '56%'"


**Should leadership publish “Late Delivery Rate = 56%” today?** Not as a bare number. Publish **“56.5% of 1,528 delivered orders arrived after the promised ETA (VP Ops definition); 23.2% were more than 10 minutes late”** with the WARN caveats (numbers after the pipeline recovers the 37 Driver App completions and drops the 4 impossible promises; 56.4% / 23.3% on the raw historical population).

**Before it becomes a clean, publishable KPI:** (1) the VP Operations signs the definition and the dedupe rule; (2) the Order Platform explains the 37 missing completion timestamps, the pre-creation promises and `pickup_at` on cancelled orders; (3) Support confirms the ticket-category mapping.

*The job was not to make the data look clean — it was to decide whether it is safe for this decision, and what remains unresolved.*